# Uzbek NER: train, predict, evaluate

Полный цикл baseline для Kaggle. Перед запуском включите `Settings -> Accelerator -> GPU` и `Internet -> On`. Запускайте ячейки сверху вниз. Результаты сохраняются в `/kaggle/working/artifacts`.

## 1. Зависимости

Kaggle уже предоставляет CUDA-сборку PyTorch, поэтому Notebook устанавливает только недостающие библиотеки проекта.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

required = {
    "transformers": "5.14.1",
    "tokenizers": "0.22.2",
    "tqdm": "4.70.0",
}
to_install = []
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        to_install.append(f"{package}=={version}")

if to_install:
    print("Installing:", ", ".join(to_install))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *to_install],
        check=True,
    )
else:
    print("Dependencies are already installed.")

## 2. Проект и окружение

In [ ]:
import hashlib
import json
import os
import platform
import shlex
import shutil
import time
from collections import Counter
from pathlib import Path

import torch

candidates = [
    Path.cwd(),
    Path.cwd() / "ner_uzb",
    Path("/kaggle/working/ner_uzb"),
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "data/train.jsonl").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Clone ner_uzb into /kaggle/working first."
    )
os.chdir(PROJECT_ROOT)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Project:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Конфигурация

`full` использует весь датасет. `medium` нужен для более короткого эксперимента. `smoke` проверяет только работоспособность и не предназначен для оценки качества.

In [ ]:
RUN_MODE = "full"  # smoke | medium | full
RUN_NAME = "baseline"
RUN_TRAIN = True
RUN_PREDICT = True
RUN_EVALUATE = True
OVERWRITE_OUTPUT = False
REQUIRE_CUDA = True

PROFILES = {
    "smoke": {"epochs": 2, "train_limit": 500, "dev_limit": 100},
    "medium": {"epochs": 3, "train_limit": 3000, "dev_limit": 300},
    "full": {"epochs": 3, "train_limit": None, "dev_limit": None},
}
TRAIN_BATCH_SIZE = 16
PREDICT_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 5e-5
MAX_LENGTH = 256
STRIDE = 64
SEED = 42
NUM_WORKERS = 2

if RUN_MODE not in PROFILES:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")
if REQUIRE_CUDA and DEVICE != "cuda":
    raise RuntimeError(
        "CUDA is unavailable. Enable GPU in Kaggle Settings, then restart the session."
    )

profile = PROFILES[RUN_MODE]
workspace = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT
OUTPUT_DIR = workspace / "artifacts" / RUN_NAME
MODEL_DIR = OUTPUT_DIR / "model"
PREDICTIONS_PATH = OUTPUT_DIR / "dev_predictions.jsonl"
METRICS_PATH = OUTPUT_DIR / "dev_metrics.json"

print(json.dumps({
    "mode": RUN_MODE,
    "run_name": RUN_NAME,
    "device": DEVICE,
    "output_dir": str(OUTPUT_DIR),
    **profile,
}, indent=2))

## 4. Проверка данных

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def summarize_jsonl(path):
    records = 0
    entities = Counter()
    empty = 0
    with Path(path).open(encoding="utf-8") as stream:
        for line in stream:
            record = json.loads(line)
            records += 1
            empty += not record["entities"]
            entities.update(entity["label"] for entity in record["entities"])
    return {"records": records, "empty": empty, "entities": dict(entities)}


manifest = json.loads(Path("data/dataset_manifest.json").read_text(encoding="utf-8"))
for split_name in ("train", "dev"):
    split = manifest["splits"][split_name]
    actual_hash = sha256(split["path"])
    if actual_hash != split["sha256"]:
        raise ValueError(f"SHA-256 mismatch for {split['path']}")
    print(split_name, summarize_jsonl(split["path"]))
print("Dataset validation: OK")

## 5. Обучение

In [ ]:
def run_command(command):
    print("$", shlex.join(map(str, command)), flush=True)
    started = time.monotonic()
    subprocess.run([str(item) for item in command], cwd=PROJECT_ROOT, check=True)
    print(f"Completed in {(time.monotonic() - started) / 60:.1f} min")


if RUN_TRAIN:
    command = [
        sys.executable, "-m", "baseline.train",
        "--train", "data/train.jsonl",
        "--dev", "data/dev.jsonl",
        "--output-dir", OUTPUT_DIR,
        "--epochs", profile["epochs"],
        "--batch-size", TRAIN_BATCH_SIZE,
        "--gradient-accumulation-steps", GRADIENT_ACCUMULATION_STEPS,
        "--learning-rate", LEARNING_RATE,
        "--max-length", MAX_LENGTH,
        "--stride", STRIDE,
        "--seed", SEED,
        "--num-workers", NUM_WORKERS,
        "--device", DEVICE,
    ]
    if profile["train_limit"] is not None:
        command.extend(["--max-train-records", profile["train_limit"]])
    if profile["dev_limit"] is not None:
        command.extend(["--max-dev-records", profile["dev_limit"]])
    if OVERWRITE_OUTPUT:
        command.append("--overwrite-output-dir")
    run_command(command)
else:
    print("Training skipped; using:", MODEL_DIR)

## 6. Предсказания на полном dev

In [ ]:
if RUN_PREDICT:
    if not MODEL_DIR.exists():
        raise FileNotFoundError(f"Model not found: {MODEL_DIR}")
    run_command([
        sys.executable, "-m", "baseline.predict",
        "--model-dir", MODEL_DIR,
        "--input", "data/dev.jsonl",
        "--output", PREDICTIONS_PATH,
        "--batch-size", PREDICT_BATCH_SIZE,
        "--device", DEVICE,
    ])
else:
    print("Prediction skipped; using:", PREDICTIONS_PATH)

## 7. Exact-span evaluation

In [ ]:
if RUN_EVALUATE:
    if not PREDICTIONS_PATH.exists():
        raise FileNotFoundError(f"Predictions not found: {PREDICTIONS_PATH}")
    run_command([
        sys.executable, "scripts/evaluate.py",
        "--gold", "data/dev.jsonl",
        "--predictions", PREDICTIONS_PATH,
        "--output", METRICS_PATH,
    ])
else:
    print("Evaluation skipped; using:", METRICS_PATH)

## 8. Результаты и архив

In [ ]:
if METRICS_PATH.exists():
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    print("Micro F1:", f"{metrics['micro']['f1']:.4f}")
    print("Macro F1:", f"{metrics['macro']['f1']:.4f}")
    for label, values in metrics["by_label"].items():
        print(
            f"{label:>4}: P={values['precision']:.4f} "
            f"R={values['recall']:.4f} F1={values['f1']:.4f}"
        )

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Output directory not found: {OUTPUT_DIR}")
archive_path = shutil.make_archive(
    str(OUTPUT_DIR.parent / RUN_NAME),
    "gztar",
    root_dir=OUTPUT_DIR,
)
print("Artifacts:", OUTPUT_DIR)
print("Archive:", archive_path)